# AgentReel: Kaggle notebook (2x T4) — setup, Live Run, review, export, viva

**Notebook settings (right panel):** Accelerator **GPU T4 x2** · Internet **On** · Persistence **Files**.
No datasets are required: the repo is cloned from GitHub and every model is pulled from Hugging Face at the
revision pinned in `config/models.yaml` (~28 GB to scratch disk `/tmp/agentreel`, ~10-20 min the first time per session).
Optional: attach `agentreel-models` / `agentreel-bin` / `agentreel-wheels` datasets (see `scripts/make_kaggle_datasets.md`) to skip downloads.

| When | Cell | What |
|---|---|---|
| Before the draw (T-45 min) | 1 | Get the repo into `/kaggle/working/agentreel` (set `REPO_URL` below) |
| | 2 | Load `TAVILY_API_KEY` (+ optional `HF_TOKEN`, `GITHUB_TOKEN`) from **Add-ons → Secrets** |
| | 3 | Disk + GPU check |
| | 4 | `scripts/setup_kaggle.sh`: pip, Python 3.12 voice venv, models from HF, Node 22 + Chrome, ffmpeg, llama.cpp, llama-server, real smoke test. Must end **ALL GREEN** |
| T+0 (card drawn) | 5 | Fill the `BRIEF` dict from the card (schema-validated → `briefs/<ID>.yaml`) |
| | 6 | Run the pipeline (output streams, which keeps the 20-min idle timer from firing) |
| A/C briefs | 7 | Review panel: approve / reject with note + target agent (mandatory, not an intervention) |
| End | 8 | Verify logs, cost report, agent graph, process report; zip outputs + logs to `/kaggle/working` |
| Viva | 9 | `--only scriptwriter` re-run + diff |

Any manual change to agent output is an **intervention**: record it with `RunLogger(ctx).intervention(stage, what, why)` (max 5, target 0).

In [ ]:
# Cell 1: get the repo
# >>> set this once you have pushed the repo to GitHub <<<
REPO_URL_DEFAULT = "https://github.com/jeet-1236/ai_agentic_Reel.git"
GIT_REF_DEFAULT = "main"            # use the tag v1.0-build / v1.0-final on Live Run day
import os, subprocess, sys, shutil, glob, json, pathlib
print(sys.version)

REPO = pathlib.Path("/kaggle/working/agentreel")
REPO_URL = os.environ.get("AGENTREEL_REPO_URL", REPO_URL_DEFAULT)
if "<your-github-user>" in REPO_URL:
    REPO_URL = ""                    # not configured -> fall back to an attached agentreel-code dataset
GIT_REF = os.environ.get("AGENTREEL_REF", GIT_REF_DEFAULT)
if (REPO / "run.py").exists():
    print("repo present:", REPO)
elif REPO_URL:
    tok = ""
    try:
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("GITHUB_TOKEN")      # only needed for a private repo
    except Exception:
        pass
    url = REPO_URL.replace("https://", f"https://x-access-token:{tok}@") if tok else REPO_URL
    subprocess.run(["git", "clone", "--branch", GIT_REF, url, str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "remote", "set-url", "origin", REPO_URL], check=True)  # drop token
else:
    src = [p for p in glob.glob("/kaggle/input/**/run.py", recursive=True) if "agentreel" in p]
    assert src, "no repo: set AGENTREEL_REPO_URL or attach the agentreel-code dataset"
    shutil.copytree(pathlib.Path(src[0]).parent, REPO, dirs_exist_ok=True)
    print("copied repo from", pathlib.Path(src[0]).parent)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print("cwd:", os.getcwd(), "| git:", subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                                                     capture_output=True, text=True).stdout.strip())

In [ ]:
# Cell 2: secrets from Kaggle Secrets (never hard-code or print them)
from kaggle_secrets import UserSecretsClient
_s = UserSecretsClient()
for name, required in (("TAVILY_API_KEY", True), ("HF_TOKEN", False), ("GITHUB_TOKEN", False)):
    try:
        os.environ[name] = _s.get_secret(name)
        print(f"{name}: loaded ({len(os.environ[name])} chars)")
    except Exception as e:
        print(f"{name}: NOT available ({e.__class__.__name__})" + ("  <-- research falls back to sitemap/ddgs" if required else ""))

In [ ]:
# Cell 3: disk + GPU check (models need ~32 GB free on the scratch disk; /kaggle/working is capped at 20 GB)
import shutil, subprocess, sys
print(sys.version)
print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,driver_version,memory.total,memory.used",
                      "--format=csv"], capture_output=True, text=True).stdout)
print("KAGGLE_KERNEL_RUN_TYPE =", os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
for path in ("/tmp", "/kaggle/working"):
    total, used, free = shutil.disk_usage(path)
    print(f"{path:16s} free {free / 1e9:6.1f} GB of {total / 1e9:6.1f} GB")
assert shutil.disk_usage("/tmp").free > 32e9, "not enough scratch disk for the models: set AGENTREEL_SCRATCH"
gpus = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.count("GPU ")
assert gpus >= 2, f"{gpus} GPU(s): set Accelerator = GPU T4 x2"

In [ ]:
# Cell 4: setup (idempotent). Cold: ~25-35 min incl. ~28 GB of models + first llama.cpp JIT; warm: <1 min.
# Must end with ALL GREEN. Knobs: SKIP_MODELS=1, WITH_FALLBACK_MODELS=1, LLAMA_TAG=..., AGENTREEL_SCRATCH=...
!mkdir -p logs && sed -i "s/\r$//" scripts/*.sh && bash scripts/setup_kaggle.sh 2>&1 | tee logs/setup.log

# load the env the setup script exported (PATH extras, model dirs, Chrome path) into this kernel
env_file = REPO / ".setup" / "env"
if env_file.exists():
    for line in env_file.read_text().splitlines():
        if "=" in line:
            k, v = line.split("=", 1)
            os.environ[k] = v
for extra in (REPO / "bin" / "node" / "bin", REPO / "bin" / "llama"):
    if extra.exists() and str(extra) not in os.environ["PATH"]:
        os.environ["PATH"] = f"{extra}:{os.environ['PATH']}"
from core.servers import health, platform_context, ensure_llm
print("llama-server healthy:", health(platform_context().llm_base_url))

In [ ]:
# Cell 5: the brief card -> briefs/<ID>.yaml (fill on Live Run day; schema-validated)
import yaml
from core.schemas import Brief

BRIEF = {
    "id": "",                  # e.g. "A03" (first letter = category A-E)
    "category": "",            # as printed on the card
    "topic": "",
    "audience": "",
    "format": "video_9x16",    # video_9x16 | video_16x9
    "duration_s": 45,          # 30-60
    "tone": "",
    "language": "English",
    "must_include": [],
    "must_avoid": [],
    "cta": "",
}
SEED = 42
brief = Brief.model_validate(BRIEF)
assert brief.id and brief.topic, "fill BRIEF first"
BRIEF_PATH = REPO / "briefs" / f"{brief.id}.yaml"
BRIEF_PATH.write_text(yaml.safe_dump(brief.model_dump(), sort_keys=False, allow_unicode=True), encoding="utf-8")
RUN_ID = f"{brief.id}-s{SEED}"
print(BRIEF_PATH.read_text())
print("category code:", brief.category_code, "| mandatory review:", brief.category_code in "AC", "| run id:", RUN_ID)

In [ ]:
# Cell 6: run the pipeline (streams progress -> keeps the session active)
import time

def stream(cmd: list[str]) -> int:
    # Run a command, echo its output line by line (with elapsed time), return the exit code.
    t0 = time.time()
    print("$", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(f"[{(time.time() - t0) / 60:5.1f}m] {line}", end="", flush=True)
    rc = p.wait()
    print(f"exit {rc} after {(time.time() - t0) / 60:.1f} min", flush=True)
    return rc

ensure_llm(platform_context(SEED, RUN_ID))            # no-op if already healthy
# equivalent to: !python run.py --brief {BRIEF_PATH} --seed {SEED}
RC = stream([sys.executable, "run.py", "--brief", str(BRIEF_PATH), "--seed", str(SEED)])
print({0: "DONE", 3: "WAITING FOR REVIEW -> use cell 7"}.get(RC, "FAILED -> check logs/run_log.jsonl"))

In [ ]:
# Cell 7: review panel (A/C mandatory review; also usable for B/D/E inspection)
import ipywidgets as W
from IPython.display import display, Markdown, Video, clear_output

out_dir = REPO / "outputs" / RUN_ID
review_dir = out_dir / "review"

def _preview():
    vids = sorted(review_dir.glob("*.mp4")) or sorted(out_dir.glob("*.mp4"))
    return max(vids, key=lambda p: p.stat().st_mtime) if vids else None

panel_out = W.Output()
note = W.Textarea(placeholder="Reject note: what must change (required for reject)", layout=W.Layout(width="100%", height="80px"))
target = W.Dropdown(options=["scriptwriter", "researcher", "storyboarder", "asset_generator", "editor"],
                    value="scriptwriter", description="target")
approve_btn = W.Button(description="Approve", button_style="success", icon="check")
reject_btn = W.Button(description="Reject", button_style="danger", icon="times")
log_out = W.Output()

def show_packet():
    with panel_out:
        clear_output()
        pending = review_dir / "PENDING_REVIEW.json"
        print("pending review:", pending.exists())
        packet = review_dir / "packet.md"
        display(Markdown(packet.read_text(encoding="utf-8") if packet.exists() else "_no review/packet.md yet_"))
        v = _preview()
        if v:
            print("preview:", v.relative_to(REPO), f"{v.stat().st_size / 1e6:.1f} MB")
            display(Video(str(v), embed=v.stat().st_size < 60e6, width=360))

def _resume(args):
    approve_btn.disabled = reject_btn.disabled = True
    with log_out:
        clear_output()
        rc = stream([sys.executable, "run.py", "--resume", RUN_ID, *args])
        print({0: "DONE", 3: "waiting for another review -> panel refreshed"}.get(rc, "FAILED"))
    approve_btn.disabled = reject_btn.disabled = False
    show_packet()

approve_btn.on_click(lambda _: _resume(["--approve"]))
def _reject(_):
    if not note.value.strip():
        with log_out:
            print("write a note first")
        return
    _resume(["--reject", note.value.strip(), "--target", target.value])
reject_btn.on_click(_reject)

show_packet()
display(W.VBox([panel_out, note, W.HBox([target, approve_btn, reject_btn]), log_out]))

In [ ]:
# Cell 8: verify + reports + export (zip to /kaggle/working for download)
import datetime
for cmd in (["scripts/verify_logs.py"], ["scripts/make_cost_report.py"],
            ["scripts/make_agent_graph.py"], ["scripts/make_process_report.py", "--run-id", RUN_ID]):
    stream([sys.executable, *cmd])

stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M")
bundle = pathlib.Path(f"/kaggle/working/export_{RUN_ID}_{stamp}")
bundle.mkdir(parents=True, exist_ok=True)
shutil.copytree(REPO / "outputs" / RUN_ID, bundle / "outputs" / RUN_ID, dirs_exist_ok=True,
                ignore=shutil.ignore_patterns("workers", "checkpoints.sqlite*"))
shutil.copytree(REPO / "logs", bundle / "logs", dirs_exist_ok=True)
for f in ("docs/agent_graph.mmd", "docs/agent_graph.png", "docs/process_report.md", "docs/process_report.pdf",
          "data/sources_manifest.csv", "config/models.lock.yaml"):
    if (REPO / f).exists():
        (bundle / f).parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(REPO / f, bundle / f)
zip_path = shutil.make_archive(str(bundle), "zip", bundle)
print("bundle:", zip_path, f"{os.path.getsize(zip_path) / 1e6:.1f} MB")
print("final files:", sorted(p.name for p in (REPO / "outputs" / RUN_ID).glob("*_final*")))

In [ ]:
# Cell 9: viva - re-run exactly one stage from cache and diff its outputs
import difflib, hashlib
STAGE = "scriptwriter"
out_dir = REPO / "outputs" / RUN_ID

def snapshot() -> dict:
    return {p.name: p.read_text(encoding="utf-8") for p in out_dir.glob("*.json")}

before = snapshot()
stream([sys.executable, "run.py", "--brief", str(BRIEF_PATH), "--seed", str(SEED), "--only", STAGE])
after = snapshot()
changed = [k for k in after if before.get(k) != after[k]]
print("changed files:", changed or "none (identical re-run: same seed + cache key)")
for k in changed:
    diff = difflib.unified_diff(before.get(k, "").splitlines(), after[k].splitlines(), f"before/{k}", f"after/{k}", lineterm="")
    print("\n".join(list(diff)[:200]))